# Apprentissage Non Supervisé - Performance des Étudiants

**Dataset : student-mat.csv**

Données sur des étudiants en mathématiques avec leurs caractéristiques socio-démographiques, habitudes et résultats scolaires.

---

## Configuration et Imports

In [ ]:
from utils_ans import *
setup_environment()

---
# I. Réduction de dimensions et Visualisation des données
---

## 1. Importation du jeu de données

In [ ]:
data = load_data('./data/student-mat.csv')

In [ ]:
# Description des variables
variables_desc = {
    'sex': 'Sexe (0=F, 1=M)',
    'age': 'Âge',
    'Medu': 'Éducation mère (0-4)',
    'Fedu': 'Éducation père (0-4)',
    'studytime': 'Temps étude hebdo (1-4)',
    'failures': 'Nombre échecs passés',
    'goout': 'Sorties avec amis (1-5)',
    'Dalc': 'Conso alcool semaine (1-5)',
    'Walc': 'Conso alcool weekend (1-5)',
    'absences': 'Nombre absences',
    'Results': 'Note finale'
}
for var, desc in variables_desc.items():
    print(f"  {var:12} : {desc}")

In [ ]:
data.describe()

In [ ]:
# Features : toutes les colonnes sauf Id
feature_cols = ['sex', 'age', 'address', 'famsize', 'Medu', 'Fedu', 'traveltime', 
                'studytime', 'failures', 'schoolsup', 'activities', 'internet', 
                'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 
                'health', 'absences', 'Results']

X, labels, feature_names = prepare_data(data, feature_cols=feature_cols, label_col='Id')

## 2. Analyse en Composantes Principales (ACP)

In [ ]:
X_scaled, X_pca, pca, scaler = perform_pca(X)

In [ ]:
cumulative = print_variance_explained(pca)

In [ ]:
plot_variance(pca)

### Réponse : Nombre d'axes à retenir

Les données étudiantes sont très hétérogènes :
- PC1+PC2 : ~25% de variance seulement
- Il faut ~10 composantes pour 80% de variance

**Conclusion :** Pour la visualisation on utilise 2 axes, mais beaucoup d'information est perdue.

In [ ]:
loadings_df = get_loadings(pca, feature_names)

### Interprétation des axes

**PC1 - "Comportement social"** : corrélé avec Dalc, Walc, goout

**PC2 - "Milieu socio-économique"** : corrélé avec Medu, Fedu, Results

In [ ]:
plot_correlation_circle(pca, feature_names)

In [ ]:
plt.figure(figsize=(12, 10))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=data['Results'], 
                      cmap='RdYlGn', s=50, alpha=0.7)
plt.colorbar(scatter, label='Note (Results)')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('Projection des étudiants (coloré par les notes)')
plt.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
plt.axvline(x=0, color='k', linestyle='-', linewidth=0.5)
plt.grid(True, alpha=0.3)
plt.show()

---
# II. Clustering
---

## 1. KMeans (3 clusters)

In [ ]:
clustering_kmeans, kmeans = apply_kmeans(X_scaled, n_clusters=3)

print("Répartition :")
for i in range(3):
    print(f"Cluster {i}: {np.sum(clustering_kmeans == i)} étudiants")

In [ ]:
print("Profil moyen de chaque cluster :")
for i in range(3):
    mask = clustering_kmeans == i
    cluster_data = data[mask]
    print(f"\nCluster {i} ({mask.sum()} étudiants):")
    print(f"  Note moyenne:    {cluster_data['Results'].mean():.1f}")
    print(f"  Temps étude:     {cluster_data['studytime'].mean():.1f}/4")
    print(f"  Échecs passés:   {cluster_data['failures'].mean():.2f}")
    print(f"  Alcool weekend:  {cluster_data['Walc'].mean():.1f}/5")
    print(f"  Sorties:         {cluster_data['goout'].mean():.1f}/5")

In [ ]:
plot_clustering(X_pca, clustering_kmeans, labels, 'KMeans (K=3) - Étudiants', annotate=False)

## 2. AgglomerativeClustering

In [ ]:
clustering_single = apply_agglomerative(X_scaled, n_clusters=3, linkage='single')
plot_clustering(X_pca, clustering_single, labels, 'Agglomerative - Single', annotate=False)

In [ ]:
clustering_ward = apply_agglomerative(X_scaled, n_clusters=3, linkage='ward')
plot_clustering(X_pca, clustering_ward, labels, 'Agglomerative - Ward', annotate=False)

In [ ]:
clustering_average = apply_agglomerative(X_scaled, n_clusters=3, linkage='average')
plot_clustering(X_pca, clustering_average, labels, 'Agglomerative - Average', annotate=False)

## 3. Détermination du nombre optimal (Silhouette)

In [ ]:
scores, best_k = compute_silhouette_scores(X_scaled)

In [ ]:
plot_silhouette_scores(scores)

## 4. Comparaison des méthodes

In [ ]:
results = compare_methods(X_scaled, n_clusters=3)

## 5. Avantages et inconvénients

### Classification Hiérarchique (AgglomerativeClustering)

**Avantages :**
- Pas besoin de spécifier K à l'avance
- Produit une hiérarchie complète (dendrogramme)
- Résultats déterministes
- Peut découvrir des clusters de formes arbitraires (avec single linkage)

**Inconvénients :**
- Complexité O(n²) ou O(n³)
- Ne passe pas à l'échelle pour grands datasets
- Décisions de fusion irréversibles
- Single linkage : effet de chaînage

---

### Partitionnement (KMeans)

**Avantages :**
- Très rapide O(n×K×iterations)
- Scalable pour grands datasets
- Clusters compacts et bien séparés
- Simple à comprendre et implémenter

**Inconvénients :**
- Nécessite K à l'avance
- Sensible à l'initialisation (non-déterministe)
- Sensible aux outliers
- Assume des clusters sphériques

## 6. Approche Hybride

In [ ]:
clustering_hyb, centers = clustering_hybride(X_scaled, n_clusters=3)
score_hyb = metrics.silhouette_score(X_scaled, clustering_hyb, metric='euclidean')
print(f"Approche Hybride - Silhouette : {score_hyb:.4f}")

In [ ]:
plot_clustering(X_pca, clustering_hyb, labels, 'Approche Hybride (Ward + KMeans)', annotate=False)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
variables_to_plot = ['Results', 'studytime', 'failures', 'Walc', 'goout', 'absences']
titles = ['Notes', 'Temps étude', 'Échecs', 'Alcool WE', 'Sorties', 'Absences']

for idx, (var, title) in enumerate(zip(variables_to_plot, titles)):
    ax = axes[idx // 3, idx % 3]
    data_by_cluster = [data[clustering_hyb == i][var] for i in range(3)]
    bp = ax.boxplot(data_by_cluster, patch_artist=True)
    colors_box = ['lightcoral', 'lightblue', 'lightgreen']
    for patch, color in zip(bp['boxes'], colors_box):
        patch.set_facecolor(color)
    ax.set_xlabel('Cluster')
    ax.set_ylabel(var)
    ax.set_title(title)

plt.tight_layout()
plt.show()

---
# Conclusion

1. **ACP** : Données hétérogènes, ~10 composantes pour 80% de variance
2. **Interprétation** : PC1 = comportement social, PC2 = milieu socio-économique
3. **Clustering** : Profils d'étudiants identifiables
   - Étudiants studieux avec bonnes notes
   - Étudiants moyens
   - Étudiants en difficulté
4. **Applications** : Identification précoce des étudiants à risque